# 🖼️ AI-Powered Contextual Alt Text Generator
Generates **SEO-friendly, contextual alt text** for product images using Claude's vision API.

### How it works:
1. Reads `image_url` and `page_url` from your CSV
2. Fetches each image and sends it to Claude with page context
3. Generates natural, non-repetitive alt text relevant to the niche
4. Saves results to a new CSV with the `alt_text` column filled


## Step 1: Install Dependencies

In [ ]:
!pip install anthropic pandas requests Pillow -q

## Step 2: Set Your Anthropic API Key
Get your key from https://console.anthropic.com → API Keys

Add it via the 🔑 **Secrets** icon in the left sidebar (name: `ANTHROPIC_API_KEY`)

In [ ]:
import os
from google.colab import userdata

try:
    ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')
    print('✅ API key loaded from Colab Secrets')
except Exception:
    ANTHROPIC_API_KEY = 'sk-ant-your-key-here'  # ← Replace if not using Secrets
    print('⚠️  Using manually pasted key')

os.environ['ANTHROPIC_API_KEY'] = ANTHROPIC_API_KEY


## Step 3: Upload Your CSV
Upload your CSV with `image_url` and `page_url` columns.

In [ ]:
from google.colab import files
import pandas as pd
import io

uploaded = files.upload()
filename = list(uploaded.keys())[0]
df = pd.read_csv(io.BytesIO(uploaded[filename]))
df.columns = [c.strip().lower().replace(' ', '_') for c in df.columns]
print(f'✅ Loaded {len(df)} rows | Columns: {list(df.columns)}')
df.head()


## Step 4: Auto-Fix Swapped Columns
Some rows may have `image_url` and `page_url` swapped — this cell corrects them.

In [ ]:
import re

IMAGE_EXT = re.compile(r'\.(jpg|jpeg|png|webp|gif|bmp|svg)(\?.*)?$', re.IGNORECASE)

def is_image_url(url):
    return bool(IMAGE_EXT.search(str(url)))

swapped = 0
for idx, row in df.iterrows():
    img = str(row.get('image_url', ''))
    page = str(row.get('page_url', ''))
    if not is_image_url(img) and is_image_url(page):
        df.at[idx, 'image_url'] = page
        df.at[idx, 'page_url'] = img
        swapped += 1

print(f'🔄 Auto-corrected {swapped} swapped rows')
df[['image_url', 'page_url']].head(10)


## Step 5: Define Alt Text Generator

In [ ]:
import anthropic
import requests
import base64
import time
from urllib.parse import urlparse

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

HEADERS = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}

def fetch_image_as_base64(url, timeout=15):
    try:
        r = requests.get(url, headers=HEADERS, timeout=timeout)
        r.raise_for_status()
        ct = r.headers.get('Content-Type', 'image/jpeg').split(';')[0].strip()
        if 'webp' in ct: ct = 'image/webp'
        elif 'png' in ct: ct = 'image/png'
        elif 'gif' in ct: ct = 'image/gif'
        else: ct = 'image/jpeg'
        return base64.standard_b64encode(r.content).decode('utf-8'), ct
    except Exception as e:
        return None, str(e)

def extract_page_context(page_url):
    try:
        path = urlparse(page_url).path
        slug = path.strip('/').split('/')[-1]
        slug = re.sub(r'\.(html|php|asp)$', '', slug)
        return slug.replace('-', ' ').replace('_', ' ').title()
    except Exception:
        return ''

PROMPT = """You are an SEO and accessibility expert in industrial B2B product photography.

Page context: {page_context}
Page URL: {page_url}

Write a single concise alt text for the image above.

Rules:
- 8 to 15 words maximum
- Start with the main visual element (product type, material, shape)
- Incorporate niche naturally, no keyword stuffing
- Describe visible details: finish, shape, orientation
- No repeated words
- Do NOT start with Image of, Photo of, or Picture of
- No brand names unless visible
- Sentence case only
- Return ONLY the alt text, no quotes or explanation"""

def generate_alt_text(image_url, page_url):
    ctx = extract_page_context(page_url)
    b64, media_type = fetch_image_as_base64(image_url)
    if b64 is None:
        return f'[ERROR fetching image: {media_type}]'
    try:
        msg = client.messages.create(
            model='claude-sonnet-4-20250514',
            max_tokens=60,
            messages=[{
                'role': 'user',
                'content': [
                    {'type': 'image', 'source': {'type': 'base64', 'media_type': media_type, 'data': b64}},
                    {'type': 'text', 'text': PROMPT.format(page_context=ctx, page_url=page_url)}
                ]
            }]
        )
        return msg.content[0].text.strip()
    except Exception as e:
        return f'[API ERROR: {e}]'

print('✅ Functions ready')


## Step 6: Run the Generator
Processes all rows with live progress. Auto-saves every 5 rows.

In [ ]:
OUTPUT_FILE = 'alt_text_results.csv'
DELAY = 0.8

if 'alt_text' not in df.columns:
    df['alt_text'] = ''

total = len(df)

for i, (idx, row) in enumerate(df.iterrows(), start=1):
    existing = str(row.get('alt_text', '')).strip()
    if existing and not existing.startswith('['):
        print(f'[{i}/{total}] ⏭️  Skipping (done): {str(row["image_url"])[:60]}')
        continue

    img_url = str(row['image_url']).strip()
    page_url = str(row['page_url']).strip()

    if not img_url or img_url == 'nan':
        df.at[idx, 'alt_text'] = '[SKIPPED: missing image URL]'
        continue

    print(f'[{i}/{total}] 🔍 {img_url[:70]}')
    alt = generate_alt_text(img_url, page_url)
    df.at[idx, 'alt_text'] = alt
    print(f'         ✅ {alt}')

    if i % 5 == 0:
        df.to_csv(OUTPUT_FILE, index=False)
        print(f'         💾 Autosaved ({i}/{total})')

    time.sleep(DELAY)

df.to_csv(OUTPUT_FILE, index=False)
print(f'\n🎉 Done! {total} rows processed.')


## Step 7: Preview Results

In [ ]:
from IPython.display import display

preview = df[['image_url', 'page_url', 'alt_text']].copy()
preview['image_url'] = preview['image_url'].str[-55:]
preview['page_url'] = preview['page_url'].str[-55:]
display(preview)

errors = df['alt_text'].str.startswith('[').sum()
print(f'\n📊 Total: {len(df)} | Success: {len(df)-errors} | Errors: {errors}')


## Step 8: Download Results

In [ ]:
from google.colab import files
df.to_csv(OUTPUT_FILE, index=False)
files.download(OUTPUT_FILE)
print('📥 Downloading alt_text_results.csv ...')


## (Optional) Step 9: Retry Failed Rows

In [ ]:
error_mask = df['alt_text'].str.startswith('[', na=True)
retry_df = df[error_mask]
print(f'🔁 Retrying {len(retry_df)} failed rows...')

for idx, row in retry_df.iterrows():
    img_url = str(row['image_url']).strip()
    page_url = str(row['page_url']).strip()
    print(f'   Retrying: {img_url[:70]}')
    alt = generate_alt_text(img_url, page_url)
    df.at[idx, 'alt_text'] = alt
    print(f'   ✅ {alt}')
    time.sleep(1.5)

df.to_csv(OUTPUT_FILE, index=False)
print('✅ Retry complete. File saved.')
